# Prompt Robustness Tables for GPT-5.1

This notebook is intentionally limited to the generation of three supplement-ready tables:

- Technical workflow summary
- Biological output summary
- Seed/Output Nodes choice 

The code below loads the prompt-robustness runs, extracts the required metadata from the chat logs and model artifacts, and exports only these three tables.

In [56]:
from __future__ import annotations

import json
import re
import textwrap
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

CWD = Path.cwd()
if (CWD / "general_prompt").exists() and (CWD / "specific_prompt").exists():
    ROOT = CWD
    PROJECT_ROOT = CWD.parent
elif (CWD / "prompt_robustness_analysis" / "general_prompt").exists():
    PROJECT_ROOT = CWD
    ROOT = CWD / "prompt_robustness_analysis"
else:
    raise FileNotFoundError(
        "Could not locate prompt_robustness_analysis from the current notebook working directory."
    )

PROMPT_TYPES = ["general_prompt", "specific_prompt"]
RUN_ORDER = ["run_1", "run_2", "run_3"]
PROMPT_LABELS = {
    "general_prompt": "General prompt",
    "specific_prompt": "Specific prompt",
}
OUT_DIR = PROJECT_ROOT / "outputs/metrics/prompt_robustness"
FIG_DIR = PROJECT_ROOT / "outputs/figures/prompt_robustness"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

CONFIRMATIONS = {"yes", "yes please", "ok", "okay", "proceed", "go ahead"}
BOOLEAN_KEYWORDS = {"and", "or", "not", "true", "false"}


def discover_runs():
    rows = []
    for prompt_type in PROMPT_TYPES:
        base_dir = ROOT / prompt_type
        seen = set()
        for run_name in RUN_ORDER:
            seen.add(run_name)
            rows.append({
                "prompt_type": prompt_type,
                "prompt_label": PROMPT_LABELS[prompt_type],
                "run_name": run_name,
                "run_dir": base_dir / run_name,
            })
        if base_dir.exists():
            for run_dir in sorted(base_dir.glob("run_*")):
                if run_dir.name in seen:
                    continue
                rows.append({
                    "prompt_type": prompt_type,
                    "prompt_label": PROMPT_LABELS[prompt_type],
                    "run_name": run_dir.name,
                    "run_dir": run_dir,
                })
    return rows


def load_json(path: Path):
    if not path.exists():
        return None
    with path.open() as handle:
        return json.load(handle)


def load_chat_requests(chat_path: Path):
    data = load_json(chat_path)
    if data is None:
        return []
    if isinstance(data, dict):
        return data.get("requests", [])
    return data


def extract_message_text(message: dict):
    if not isinstance(message, dict):
        return "", False
    parts = message.get("parts", [])
    texts = []
    is_agent = False
    for part in parts:
        if not isinstance(part, dict):
            continue
        if "agent" in part:
            is_agent = True
        text = part.get("text")
        if isinstance(text, str) and text.strip():
            texts.append(text.strip())
    combined = " ".join(texts).strip()
    return combined, is_agent


def classify_intervention(text: str, is_agent: bool):
    normalized = text.lower().strip()
    if "try again" in normalized:
        return "error_retry"
    if "continue" in normalized or is_agent:
        return "max_tools_continue"
    if normalized in CONFIRMATIONS:
        return "confirmation"
    return "short_prompt"


def iter_tool_events(requests):
    for request in requests:
        response_items = request.get("response", []) if isinstance(request, dict) else []
        for item in response_items:
            if isinstance(item, dict) and item.get("kind") == "toolInvocationSerialized":
                yield item


def collect_timestamps(obj, timestamps=None):
    if timestamps is None:
        timestamps = []
    if isinstance(obj, dict):
        for key, value in obj.items():
            if key == "timestamp" and isinstance(value, (int, float)):
                timestamps.append(value)
            else:
                collect_timestamps(value, timestamps)
    elif isinstance(obj, list):
        for item in obj:
            collect_timestamps(item, timestamps)
    return timestamps


def parse_bnet_network(bnet_path: Path):
    nodes = set()
    edges = set()
    if not bnet_path.exists():
        return nodes, edges

    for raw_line in bnet_path.read_text().splitlines():
        line = raw_line.split("#", 1)[0].strip()
        if not line or "," not in line:
            continue
        target, expr = line.split(",", 1)
        target = target.strip()
        if not target:
            continue
        nodes.add(target)
        regulators = {
            token
            for token in re.findall(r"[A-Za-z_][A-Za-z0-9_]*", expr)
            if token.lower() not in BOOLEAN_KEYWORDS
        }
        nodes.update(regulators)
        edges.update((source, target) for source in regulators)

    return nodes, edges


def parse_result_distribution(result_path: Path):
    if not result_path.exists():
        return []
    raw = pd.read_csv(result_path, header=None)
    if raw.empty:
        return []

    states = [str(value).strip() for value in raw.iloc[0].tolist()]
    if len(raw.index) > 1:
        probs = pd.to_numeric(raw.iloc[1], errors="coerce").tolist()
    else:
        probs = [np.nan] * len(states)

    rows = []
    for state, prob in zip(states, probs):
        rows.append({"state": state, "probability": prob})

    rows.sort(
        key=lambda row: (
            pd.isna(row["probability"]),
            -(row["probability"] if pd.notna(row["probability"]) else -1),
        )
    )
    return rows


def extract_event_output_texts(event: dict):
    texts = []
    for output in event.get("resultDetails", {}).get("output", []):
        if isinstance(output, dict):
            value = output.get("value")
            if isinstance(value, str):
                texts.append(value)
    return texts


def extract_raw_input(event: dict):
    raw_input = event.get("toolSpecificData", {}).get("rawInput")
    if isinstance(raw_input, dict):
        return raw_input

    input_text = event.get("resultDetails", {}).get("input")
    if isinstance(input_text, str):
        try:
            parsed = json.loads(input_text)
            if isinstance(parsed, dict):
                return parsed
        except json.JSONDecodeError:
            return {}
    return {}


def normalize_node_list(value):
    if value is None:
        return []
    if isinstance(value, str):
        return [value]
    if isinstance(value, (list, tuple, set)):
        return [str(item) for item in value]
    return [str(value)]


def latest_network_seed_nodes(events):
    for event in reversed(events):
        if event.get("toolId") != "mcp_neko_create_network":
            continue
        raw_input = extract_raw_input(event)
        seed_nodes = raw_input.get("list_of_initial_genes", [])
        return normalize_node_list(seed_nodes)
    return []


def latest_initial_state_nodes(events):
    for event in reversed(events):
        if event.get("toolId") != "mcp_maboss_set_maboss_initial_state":
            continue
        raw_input = extract_raw_input(event)
        state_nodes = raw_input.get("nodes")
        return normalize_node_list(state_nodes)
    return []


def latest_output_nodes(events):
    for event in reversed(events):
        if event.get("toolId") != "mcp_maboss_set_maboss_output_nodes":
            continue
        raw_input = extract_raw_input(event)
        output_nodes = raw_input.get("output_nodes", [])
        return normalize_node_list(output_nodes)
    return []


def stringify_nodes(nodes):
    return ", ".join(nodes) if nodes else "N/A"


def save_dataframe_figure(
    df,
    title,
    png_path,
    pdf_path=None,
    wrap_cols=None,
    wrap_width=24,
    fig_width=18,
    body_font=12,
    header_font=13,
    title_font=16,
):
    if df.empty:
        return

    plot_df = df.copy()
    wrap_cols = wrap_cols or []
    for col in wrap_cols:
        if col in plot_df.columns:
            plot_df[col] = plot_df[col].map(
                lambda value: textwrap.fill(
                    str(value), width=wrap_width, break_long_words=False, break_on_hyphens=False
                )
            )

    row_units = []
    for _, row in plot_df.iterrows():
        max_lines = max(str(value).count("\n") + 1 for value in row)
        row_units.append(1.0 + 0.45 * (max_lines - 1))

    fig_height = max(3.5, 1.0 + sum(row_units) * 0.55)
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    ax.axis("off")

    col_width = 1 / len(plot_df.columns)
    table = ax.table(
        cellText=plot_df.values,
        colLabels=plot_df.columns,
        cellLoc="left",
        colLoc="left",
        colWidths=[col_width] * len(plot_df.columns),
        bbox=[0, 0, 1, 1],
    )
    table.auto_set_font_size(False)
    table.set_fontsize(body_font)

    header_color = "#234E70"
    for (row, col), cell in table.get_celld().items():
        cell.set_linewidth(0.5)
        if row == 0:
            cell.set_facecolor(header_color)
            cell.set_text_props(color="white", fontweight="bold", fontsize=header_font)
        else:
            cell.set_facecolor("#F7FAFC" if row % 2 == 0 else "white")
            cell.set_text_props(fontsize=body_font)

    ax.set_title(title, fontsize=title_font, fontweight="bold", pad=12)
    plt.tight_layout()
    plt.savefig(png_path, dpi=300, bbox_inches="tight")
    if pdf_path is not None:
        plt.savefig(pdf_path, bbox_inches="tight")
    plt.close(fig)


def save_metric_bar_chart(df, column, title, png_path, pdf_path=None, ylabel=None):
    chart_df = df.copy()
    chart_df = chart_df[chart_df[column].notna()].reset_index(drop=True)
    if chart_df.empty:
        return

    chart_df["Run Label"] = chart_df["Prompt Type"] + " | " + chart_df["Run"]
    colors = ["#4C78A8" if value == "General prompt" else "#F58518" for value in chart_df["Prompt Type"]]

    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.bar(chart_df["Run Label"], chart_df[column], color=colors)
    ax.set_title(title, fontweight="bold")
    ax.set_ylabel(ylabel or column)
    ax.tick_params(axis="x", rotation=35)
    ax.grid(axis="y", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig(png_path, dpi=300, bbox_inches="tight")
    if pdf_path is not None:
        plt.savefig(pdf_path, bbox_inches="tight")
    plt.close(fig)


def summarize_run(prompt_type: str, prompt_label: str, run_name: str, run_dir: Path):
    artifacts = {
        "chat.json": run_dir / "chat.json",
        "Network_1.bnet": run_dir / "Network_1.bnet",
        "output.bnd": run_dir / "output.bnd",
        "output.cfg": run_dir / "output.cfg",
        "result.csv": run_dir / "result.csv",
        "session_meta.json": run_dir / "session_meta.json",
    }
    artifact_presence = {name: path.exists() for name, path in artifacts.items()}

    requests = load_chat_requests(artifacts["chat.json"])
    tool_events = list(iter_tool_events(requests))
    mcp_events = [event for event in tool_events if str(event.get("toolId", "")).startswith("mcp_")]
    tool_counter = Counter(event.get("toolId", "unknown") for event in mcp_events)

    timestamps = collect_timestamps(requests, [])
    wall_time_s = (max(timestamps) - min(timestamps)) / 1000 if len(timestamps) >= 2 else np.nan

    interventions = []
    for request_index, request in enumerate(requests):
        if request_index == 0:
            continue
        text, is_agent = extract_message_text(request.get("message", {}))
        kind = classify_intervention(text, is_agent)
        interventions.append({
            "Prompt Type": prompt_label,
            "Run": run_name,
            "Request #": request_index + 1,
            "Intervention Type": kind,
            "Message": text if text else "[button click - no text]",
        })

    nodes, edges = parse_bnet_network(artifacts["Network_1.bnet"])
    result_distribution = parse_result_distribution(artifacts["result.csv"])
    seed_nodes = latest_network_seed_nodes(mcp_events)
    initial_state_nodes = latest_initial_state_nodes(mcp_events)
    output_nodes = latest_output_nodes(mcp_events)

    seed_set = set(seed_nodes)
    initial_state_set = set(initial_state_nodes)
    output_set = set(output_nodes)
    seed_output_overlap = sorted(seed_set & output_set)
    initial_output_overlap = sorted(initial_state_set & output_set)

    maboss_success = artifact_presence["result.csv"] or any(
        "maboss simulation completed successfully" in text.lower()
        for event in mcp_events
        for text in extract_event_output_texts(event)
    )
    has_run_data = any(artifact_presence.values()) or bool(mcp_events)
    workflow_completed = all([
        artifact_presence["Network_1.bnet"],
        artifact_presence["output.bnd"],
        artifact_presence["output.cfg"],
        bool(output_nodes),
        maboss_success,
    ])

    top_state = result_distribution[0]["state"] if result_distribution else "N/A"
    top_prob = result_distribution[0]["probability"] if result_distribution else np.nan

    summary = {
        "Prompt Type": prompt_label,
        "Run": run_name,
        "Has Run Data": has_run_data,
        "Run Directory Exists": run_dir.exists(),
        "Workflow Completed": "Yes" if workflow_completed else "No",
        "MaBoSS Simulation Successful": "Yes" if maboss_success else "No",
        "Wall Time (s)": wall_time_s if has_run_data else np.nan,
        "Wall Time (min)": wall_time_s / 60 if has_run_data and pd.notna(wall_time_s) else np.nan,
        "MCP Tool Calls": len(mcp_events) if has_run_data else np.nan,
        "Unique MCP Tools": len(tool_counter) if has_run_data else np.nan,
        "User Interventions": len(interventions) if has_run_data else np.nan,
        "Nodes": len(nodes) if has_run_data else np.nan,
        "Edges": len(edges) if has_run_data else np.nan,
        "Network Seed Nodes": stringify_nodes(seed_nodes),
        "Network Seed Count": len(seed_nodes) if has_run_data else np.nan,
        "MaBoSS Initial-State Nodes": stringify_nodes(initial_state_nodes),
        "Initial-State Node Count": len(initial_state_nodes) if initial_state_nodes else (0 if has_run_data else np.nan),
        "Chosen Output Nodes": stringify_nodes(output_nodes),
        "Output Node Count": len(output_nodes) if has_run_data else np.nan,
        "Seed/Output Overlap Count": len(seed_output_overlap) if has_run_data else np.nan,
        "Seed/Output Overlap Nodes": stringify_nodes(seed_output_overlap),
        "Initial-State/Output Overlap Count": len(initial_output_overlap) if has_run_data else np.nan,
        "Initial-State/Output Overlap Nodes": stringify_nodes(initial_output_overlap),
        "Top MaBoSS State": top_state,
        "Top State Probability": top_prob,
        "Artifacts Present": ", ".join(name for name, exists in artifact_presence.items() if exists) or "None",
        "Tool Call Breakdown": ", ".join(f"{tool}:{count}" for tool, count in sorted(tool_counter.items())) or "N/A",
    }

    result_rows = [
        {
            "Prompt Type": prompt_label,
            "Run": run_name,
            "State": row["state"],
            "Probability": row["probability"],
        }
        for row in result_distribution
    ]

    payload = {
        "prompt_type": prompt_type,
        "prompt_label": prompt_label,
        "run_name": run_name,
        "run_dir": run_dir,
        "nodes": nodes,
        "edges": edges,
        "seed_nodes": seed_nodes,
        "initial_state_nodes": initial_state_nodes,
        "output_nodes": output_nodes,
        "artifact_presence": artifact_presence,
        "interventions": interventions,
        "result_rows": result_rows,
    }
    return summary, payload

In [57]:
from matplotlib_venn import venn2


def _wrap_headers(columns, width=16):
    return [
        textwrap.fill(str(col), width=width, break_long_words=False, break_on_hyphens=False)
        for col in columns
    ]


def _max_line_length(value):
    lines = str(value).split("\n")
    return max((len(line) for line in lines), default=1)


def _line_count(value):
    return max(1, str(value).count("\n") + 1)


def _infer_col_widths(df, column_width_overrides=None):
    column_width_overrides = column_width_overrides or {}
    weights = []
    for col in df.columns:
        if col in column_width_overrides:
            weights.append(float(column_width_overrides[col]))
            continue
        values = [str(col)] + [str(value) for value in df[col].tolist()]
        longest = max(_max_line_length(value) for value in values)
        weights.append(max(4.0, min(float(longest), 28.0)))
    total = sum(weights) or 1.0
    return [weight / total for weight in weights]


def _set_table_row_heights(table, plot_df, wrapped_headers, line_spacing=1.15, header_line_spacing=1.1):
    header_units = 1.2 + 0.45 * (max(_line_count(value) for value in wrapped_headers) - 1)
    body_units = []
    for _, row in plot_df.iterrows():
        max_lines = max(_line_count(value) for value in row)
        body_units.append(1.0 + 0.65 * (max_lines - 1))

    total_units = header_units + sum(body_units)
    if total_units <= 0:
        return

    normalized_heights = [header_units / total_units] + [units / total_units for units in body_units]
    ncols = len(plot_df.columns)

    for row_idx, row_height in enumerate(normalized_heights):
        for col_idx in range(ncols):
            cell = table[(row_idx, col_idx)]
            cell.set_height(row_height)
            cell.PAD = 0.024 if row_idx == 0 else 0.03
            cell.get_text().set_va("center")
            cell.get_text().set_ha("left")
            cell.get_text().set_linespacing(header_line_spacing if row_idx == 0 else line_spacing)


def save_dataframe_figure(
    df,
    title,
    png_path,
    pdf_path=None,
    wrap_cols=None,
    wrap_width=20,
    fig_width=11.2,
    body_font=8.5,
    header_font=9,
    title_font=13,
    header_wrap_width=16,
    column_width_overrides=None,
    min_fig_height=2.8,
    row_height_scale=0.46,
    line_spacing=1.15,
    header_line_spacing=1.1,
):
    if df.empty:
        return

    plot_df = df.copy()
    wrap_cols = wrap_cols or []
    for col in wrap_cols:
        if col in plot_df.columns:
            plot_df[col] = plot_df[col].map(
                lambda value: textwrap.fill(
                    str(value), width=wrap_width, break_long_words=False, break_on_hyphens=False
                )
            )

    wrapped_headers = _wrap_headers(plot_df.columns, width=header_wrap_width)
    col_widths = _infer_col_widths(plot_df, column_width_overrides=column_width_overrides)

    header_units = 1.2 + 0.45 * (max(_line_count(value) for value in wrapped_headers) - 1)
    body_units = []
    for _, row in plot_df.iterrows():
        max_lines = max(_line_count(value) for value in row)
        body_units.append(1.0 + 0.65 * (max_lines - 1))

    fig_height = max(min_fig_height, 0.8 + (header_units + sum(body_units)) * row_height_scale)
    fig, ax = plt.subplots(figsize=(min(fig_width, 11.2), fig_height))
    ax.axis("off")

    table = ax.table(
        cellText=plot_df.values,
        colLabels=wrapped_headers,
        cellLoc="left",
        colLoc="left",
        colWidths=col_widths,
        bbox=[0, 0, 1, 1],
    )
    table.auto_set_font_size(False)
    table.set_fontsize(body_font)
    _set_table_row_heights(
        table,
        plot_df,
        wrapped_headers,
        line_spacing=line_spacing,
        header_line_spacing=header_line_spacing,
    )

    header_color = "#234E70"
    for (row, col), cell in table.get_celld().items():
        cell.set_linewidth(0.45)
        if row == 0:
            cell.set_facecolor(header_color)
            cell.set_text_props(color="white", fontweight="bold", fontsize=header_font)
        else:
            cell.set_facecolor("#F7FAFC" if row % 2 == 0 else "white")
            cell.set_text_props(fontsize=body_font)

    ax.set_title(title, fontsize=title_font, fontweight="bold", pad=10)
    plt.tight_layout()
    plt.savefig(png_path, dpi=300, bbox_inches="tight")
    if pdf_path is not None:
        plt.savefig(pdf_path, bbox_inches="tight")
    plt.close(fig)

In [58]:
run_summaries = []
run_payloads = {}

for entry in discover_runs():
    summary, payload = summarize_run(
        prompt_type=entry["prompt_type"],
        prompt_label=entry["prompt_label"],
        run_name=entry["run_name"],
        run_dir=entry["run_dir"],
    )
    run_summaries.append(summary)
    run_payloads[(entry["prompt_type"], entry["run_name"])] = payload

summary_df = (
    pd.DataFrame(run_summaries)
    .sort_values(["Prompt Type", "Run"])
.reset_index(drop=True)
)

technical_summary_df = summary_df[
    [
        "Prompt Type",
        "Run",
        "Workflow Completed",
        "MaBoSS Simulation Successful",
        "Wall Time (s)",
        "MCP Tool Calls",
        "Unique MCP Tools",
        "User Interventions",
    ]
]

biological_summary_df = summary_df[
    [
        "Prompt Type",
        "Run",
        "Nodes",
        "Edges",
        "Network Seed Count",
        "Initial-State Node Count",
        "Output Node Count",
        "Top MaBoSS State",
        "Top State Probability",
    ]
]

display(technical_summary_df)
display(biological_summary_df)

technical_summary_path = OUT_DIR / "prompt_robustness_technical_summary.csv"
biological_summary_path = OUT_DIR / "prompt_robustness_biological_summary.csv"
technical_summary_df.to_csv(technical_summary_path, index=False)
biological_summary_df.to_csv(biological_summary_path, index=False)

save_dataframe_figure(
    technical_summary_df,
    "Technical Workflow Summary",
    FIG_DIR / "prompt_robustness_technical_summary.png",
    FIG_DIR / "prompt_robustness_technical_summary.pdf",
    fig_width=10.8,
    header_wrap_width=13,
    column_width_overrides={
        "Prompt Type": 0.9,
        "Run": 0.55,
        "Workflow Completed": 1.0,
        "MaBoSS Simulation Successful": 1.2,
        "Wall Time (s)": 0.8,
        "MCP Tool Calls": 0.8,
        "Unique MCP Tools": 0.8,
        "User Interventions": 0.8,
    },
)
save_dataframe_figure(
    biological_summary_df,
    "Network/Boolean Model Summary",
    FIG_DIR / "prompt_robustness_biological_summary.png",
    FIG_DIR / "prompt_robustness_biological_summary.pdf",
    wrap_cols=["Top MaBoSS State"],
    wrap_width=22,
    fig_width=11.0,
    body_font=8.4,
    header_font=8.8,
    header_wrap_width=14,
    column_width_overrides={
        "Prompt Type": 0.9,
        "Run": 0.55,
        "Nodes": 0.6,
        "Edges": 0.6,
        "Network Seed Count": 0.9,
        "Initial-State Node Count": 1.0,
        "Output Node Count": 0.9,
        "Top MaBoSS State": 1.6,
        "Top State Probability": 0.9,
    },
    row_height_scale=0.5,
)

print(f"Saved: {technical_summary_path}")
print(f"Saved: {biological_summary_path}")
print("Saved: outputs/figures/prompt_robustness/prompt_robustness_technical_summary.png/.pdf")
print("Saved: outputs/figures/prompt_robustness/prompt_robustness_biological_summary.png/.pdf")

,Prompt Type,Run,Workflow Completed,MaBoSS Simulation Successful,Wall Time (s),MCP Tool Calls,Unique MCP Tools,User Interventions
0,General prompt,run_1,Yes,Yes,313.816,19,16,0
1,General prompt,run_2,Yes,Yes,211.149,15,14,0
2,General prompt,run_3,Yes,Yes,197.566,15,15,0
3,Specific prompt,run_1,Yes,Yes,148.821,16,16,0
4,Specific prompt,run_2,Yes,Yes,97.932,13,13,0
5,Specific prompt,run_3,Yes,Yes,97.114,12,12,0


,Prompt Type,Run,Nodes,Edges,Network Seed Count,Initial-State Node Count,Output Node Count,Top MaBoSS State,Top State Probability
0,General prompt,run_1,89,698,20,1,8,AKT1 -- MTOR,1.0000
1,General prompt,run_2,91,728,22,0,8,<nil>,0.5000
2,General prompt,run_3,45,253,13,0,10,RELA -- TERT -- CCND1,0.5681
3,Specific prompt,run_1,30,159,8,0,3,NFKB1,0.9999
4,Specific prompt,run_2,30,159,8,0,3,NFKB1,0.9999
5,Specific prompt,run_3,30,159,8,0,3,NFKB1,0.9999


Saved: /home/mruscone/Desktop/github/Supp_mat_MCP_orchestrator/outputs/metrics/prompt_robustness/prompt_robustness_technical_summary.csv
Saved: /home/mruscone/Desktop/github/Supp_mat_MCP_orchestrator/outputs/metrics/prompt_robustness/prompt_robustness_biological_summary.csv
Saved: outputs/figures/prompt_robustness/prompt_robustness_technical_summary.png/.pdf
Saved: outputs/figures/prompt_robustness/prompt_robustness_biological_summary.png/.pdf


In [59]:
node_choice_membership_df = summary_df[
    [
        "Prompt Type",
        "Run",
        "Network Seed Nodes",
        "MaBoSS Initial-State Nodes",
        "Chosen Output Nodes",
    ]
] .copy()

node_choice_membership_df["Run"] = (
    node_choice_membership_df["Prompt Type"]
    .str.replace(" prompt", "", regex=False)
    + " | "
    + node_choice_membership_df["Run"]
)
node_choice_membership_df = node_choice_membership_df.rename(
    columns={
        "Run": "Run Label",
        "Network Seed Nodes": "Seed nodes",
        "MaBoSS Initial-State Nodes": "Initial-state nodes",
        "Chosen Output Nodes": "Output nodes",
    }
)[["Run Label", "Seed nodes", "Initial-state nodes", "Output nodes"]]

display(node_choice_membership_df)
node_choice_path = OUT_DIR / "prompt_robustness_node_choice_membership.csv"
node_choice_membership_df.to_csv(node_choice_path, index=False)

save_dataframe_figure(
    node_choice_membership_df,
    "Seed/Output Node Choice",
    FIG_DIR / "prompt_robustness_node_choice_membership_table.png",
    FIG_DIR / "prompt_robustness_node_choice_membership_table.pdf",
    wrap_cols=["Seed nodes", "Initial-state nodes", "Output nodes"],
    wrap_width=26,
    fig_width=11.2,
    body_font=9.4,
    header_font=9.2,
    header_wrap_width=13,
    column_width_overrides={
        "Run Label": 1.0,
        "Seed nodes": 2.85,
        "Initial-state nodes": 1.5,
        "Output nodes": 1.75,
    },
    min_fig_height=5.2,
    row_height_scale=0.62,
    line_spacing=1.28,
    header_line_spacing=1.15,
)

print(f"Saved: {node_choice_path}")
print("Saved: outputs/figures/prompt_robustness/prompt_robustness_node_choice_membership_table.png/.pdf")

,Run Label,Seed nodes,Initial-state nodes,Output nodes
0,General | run_1,"TNF, TNFRSF1A, TNFRSF1B, TRADD, FADD, RIPK1, R...",TNF,"CASP3, CASP8, BAX, BCL2, CCND1, MYC, AKT1, MTOR"
1,General | run_2,"TNF, TNFRSF1A, TNFRSF1B, TRADD, TRAF2, RIPK1, ...",N/A,"CASP3, CASP8, BAX, BCL2, NFKB1, RELA, MYC, CCND1"
2,General | run_3,"TNF, TNFRSF1A, TNFRSF1B, TRADD, FADD, CASP8, C...",N/A,"CASP3, CASP7, CASP8, BAX, BAD, CCND1, MYC, TER..."
3,Specific | run_1,"TNF, TNFRSF1A, TRADD, RIPK1, CASP3, NFKB1, BAX...",N/A,"CASP3, BAX, NFKB1"
4,Specific | run_2,"TNF, TNFRSF1A, TRADD, RIPK1, CASP3, NFKB1, BAX...",N/A,"CASP3, BAX, NFKB1"
5,Specific | run_3,"TNF, TNFRSF1A, TRADD, RIPK1, CASP3, NFKB1, BAX...",N/A,"CASP3, BAX, NFKB1"


Saved: /home/mruscone/Desktop/github/Supp_mat_MCP_orchestrator/outputs/metrics/prompt_robustness/prompt_robustness_node_choice_membership.csv
Saved: outputs/figures/prompt_robustness/prompt_robustness_node_choice_membership_table.png/.pdf
